## Demo Diffusion models generative AI

Author: alberto.suarez@uam.es
Date: 2025-03-02

In [6]:
%load_ext autoreload
%autoreload 2

import numpy as np

from functools import partial 

import torch
from torch.utils.data import (
    DataLoader,
    Dataset,
    Subset,
)
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchvision.transforms import functional

import diffusion_process as dfp

from diffusion_utilities import (
    plot_image_grid,
    plot_image_evolution,
    animation_images,
)


n_threads = torch.get_num_threads()
print('Number of threads: {:d}'.format(n_threads))

device ='cpu' 


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Number of threads: 6


In [7]:
batch_size = 128

data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)
data_train = data
data_loader = DataLoader(
    data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

###  Diffusion process

In [8]:
def ve_cosine_drift_coefficient(x_t, t):
    return torch.zeros_like(x_t)

def ve_cosine_diffusion_coefficient(t):
    t_req = t.clone().detach().requires_grad_(True)
    
    sigma = ve_cosine_sigma_t(t_req)
    sigma2 = sigma ** 2

    grad = torch.autograd.grad(
        outputs=sigma2.sum(),
        inputs=t_req,
        create_graph=False
    )[0]

    g2 = torch.clamp(grad, min=1.0e-12)
    return torch.sqrt(g2)

def ve_cosine_mu_t(x_0, t):
    return x_0

def ve_cosine_sigma_t(t):
    s = 0.008
    s_tensor = torch.tensor(s, device=t.device, dtype=t.dtype)

    alpha_bar = (
        torch.cos((t + s_tensor) / (1 + s_tensor) * torch.pi / 2) ** 2
        / torch.cos((s_tensor / (1 + s_tensor)) * torch.pi / 2) ** 2
    )

    sigma2 = 1.0 - alpha_bar
    sigma2 = torch.clamp(sigma2, min=1.0e-12)
    return torch.sqrt(sigma2)
    
drift_coefficient = ve_cosine_drift_coefficient
diffusion_coefficient = ve_cosine_diffusion_coefficient
mu_t = ve_cosine_mu_t
sigma_t = ve_cosine_sigma_t

diffusion_process = dfp.GaussianDiffussionProcess(
    drift_coefficient,
    diffusion_coefficient,
    mu_t,
    sigma_t,
)


In [9]:
# Define the score model

from score_model import ScoreNet

score_model = torch.nn.DataParallel(
    ScoreNet(
        marginal_prob_std=sigma_t
    )
)
score_model = score_model.to(device)

In [10]:
# Train model
from torch.optim import Adam #optimizador calcula los pesos de la red
import torchvision.transforms as transforms 
from tqdm.notebook import trange #barra de progreso


learning_rate = 1.0e-3
optimizer = Adam(score_model.parameters(), lr=learning_rate) #optimizador para actualizar 
#los parámetros del modelo

n_epochs =  35
tqdm_epoch = trange(n_epochs) #barra de procesos

for epoch in tqdm_epoch:
    avg_loss = 0.0 #inicializas para calcular la pérdida media
    num_items = 0
    for x, y in data_loader:
        x = x.to(device)    
        loss = diffusion_process.loss_function(score_model, x) #función de pérdida
        optimizer.zero_grad() #eliminas gradientes previos
        loss.backward()    
        optimizer.step()
        avg_loss += loss.item() * x.shape[0]
        num_items += x.shape[0]
        
    tqdm_epoch.set_description('Average Loss: {:5f}'.format(avg_loss / num_items))

torch.save(score_model.state_dict(), 've_cosine.pth')

  0%|          | 0/35 [00:00<?, ?it/s]

KeyboardInterrupt: 